In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

from __future__ import absolute_import, division, print_function, unicode_literals

import os
import glob
import shutil
from tqdm import tqdm

import cv2
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mimg

# Import packages for data handling
from PIL import Image
from skimage.io import imread
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from mlxtend.plotting import plot_confusion_matrix

# pytorch
import torch
import torch.nn as nn
from torch.nn.functional import softmax
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.models import mobilenet_v2

from albumentations.augmentations.blur import MotionBlur, Defocus
import albumentations as A
from albumentations.pytorch import ToTensorV2

import warnings
warnings.filterwarnings('ignore')

color = sns.color_palette()
%matplotlib inline

# Set seed nunmber to all packages
seed_number = 42
np.random.seed(seed_number)

In [ ]:
# Configuring directories

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory
root = "../input/lcc-fasd"
input_dir = os.path.join(root,"LCC_FASD")
train_dir = os.path.join(input_dir, 'LCC_FASD_training')
val_dir = os.path.join(input_dir, 'LCC_FASD_development')
test_dir = os.path.join(input_dir, 'LCC_FASD_evaluation')

dataset_dir = [dir for dir in sorted(os.listdir(input_dir)) if os.path.isdir(os.path.join(input_dir, dir))]
label_name = [subdir for subdir in sorted(os.listdir(train_dir)) if os.path.isdir(os.path.join(train_dir, subdir))]

# Printing the directory informations
print(f"Main directories\t: {os.listdir(root)}")
print(f"Dataset sub-directories\t: {dataset_dir}")
print(f"Train set directory\t: {label_name}")

In [ ]:
dir_dict = {'train': train_dir, 'val': val_dir, 'test': test_dir}
case_count, img_disp, set_length  = {}, {}, {}

for key, val in dir_dict.items():
    case_count[key] = {}
    img_disp[key] = {}
    set_count = 0
    
    for label in label_name:
        label_list = list(sorted(glob.glob(os.path.join(val, label, "*.png"))))
        if len(label_list) == 0:
            continue

        case_count[key][label] = len(label_list)
        set_count += len(label_list)
        
        select_img_id = np.random.randint(len(label_list)-1)
        # print(select_img_id)
        img_disp[key][label] = label_list[select_img_id]
        
    set_length[key] = set_count

case_count_df = pd.DataFrame(case_count)
img_disp_df = pd.DataFrame(img_disp)
print(f"Dataset summary:\n\n{case_count_df}")

In [ ]:
# Visualizing some of the data set
num_classes = len(label_name)
num_dataset = 0
for key, val in set_length.items():
  num_dataset += 1 if val > 0 else 0

f, ax = plt.subplots(num_classes, num_dataset, figsize=(num_dataset*10, 18))

for k in range(num_classes*num_dataset):
    j, i = k//num_dataset, k%num_dataset  # Image indexing
    
    img = imread(img_disp_df.iloc[j, i])
    ax[j, i].imshow(img, cmap='gray')
    ax[j, i].set_title(f"{img_disp_df.columns[i].upper()}: {img_disp_df.index[j].capitalize()}", fontsize=32)
    ax[j, i].axis('off')
    ax[j, i].set_aspect('auto')
plt.show()

**Preprocessing**

In [ ]:
# Define the data transformations for training, validation, and testing
# train_transform = transforms.Compose([
#     transforms.RandomResizedCrop(224),
#     transforms.RandomRotation(20),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=0.15),
# #     transforms.RandomZoom(0.15),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])
train_transform = A.Compose([
    A.Resize(224, 224, interpolation=cv2.INTER_CUBIC),
    A.augmentations.transforms.ISONoise(
        color_shift=(0.15, 0.35), 
        intensity=(0.1, 0.5), 
        always_apply=False, 
        p=0.05),
    A.augmentations.transforms.RandomBrightnessContrast(
        brightness_limit=(-0.2, 0.2),
        contrast_limit=(-0.2, 0.2),
        always_apply=False,
        brightness_by_max=True, p=0.125),
    A.MotionBlur(blur_limit=3, p=0.2),
    A.augmentations.transforms.ImageCompression(
        quality_lower=50,
        quality_upper=100,
        always_apply=False,
        p=0.25),
    A.augmentations.dropout.CoarseDropout(
        max_holes=24,
        max_height=8,
        max_width=8,
        min_holes=4,
        min_height=4,
        min_width=4,
        fill_value=0,
        always_apply=False,
        p=0.25),
    A.augmentations.transforms.GaussNoise(
        var_limit=(10.0, 50.0),
        mean=0,
        always_apply=False,
        p=0.2),
    A.ShiftScaleRotate(scale_limit=0.2, rotate_limit=0, shift_limit=0, p=1.0),
    A.augmentations.geometric.transforms.ShiftScaleRotate(
        shift_limit=0.1, 
        scale_limit=0.1, 
        rotate_limit=15, 
        interpolation=cv2.INTER_CUBIC, 
        border_mode=cv2.BORDER_REFLECT_101, 
        always_apply=False, 
        p=0.5),
    A.augmentations.geometric.transforms.Affine(
        scale=(0.5, 1.5), 
        translate_percent=(0.1, 0.2), 
        rotate=(-15, 15), 
        shear=(-8, 8), 
        interpolation=cv2.INTER_CUBIC, 
        always_apply=False, 
        p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# val_transform = transforms.Compose([
#     transforms.Resize(256),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# test_transform = transforms.Compose([
#     transforms.Resize(256),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])


val_transform = A.Compose([
    A.Resize(224, 224, interpolation=cv2.INTER_CUBIC),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    # To tensor
    ToTensorV2(),
])

test_transform = A.Compose([
    A.Resize(224, 224, interpolation=cv2.INTER_CUBIC),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    # To tensor
    ToTensorV2(),
])

**Prepare Datasets**

In [ ]:
def gen_df(img_dir):
    real_img_dir = os.path.join(img_dir, 'real')
    spoof_img_dir = os.path.join(img_dir, 'spoof')
    real_files = [os.path.join(real_img_dir, f) for f in os.listdir(real_img_dir) if os.path.isfile(os.path.join(real_img_dir, f))]
    spoof_files = [os.path.join(spoof_img_dir, f) for f in os.listdir(spoof_img_dir) if os.path.isfile(os.path.join(spoof_img_dir, f))]

    # Create DataFrame
    data = {
        'file_path': real_files + spoof_files,
        'live': [1] * len(real_files) + [0] * len(spoof_files)
    }
    
    df = pd.DataFrame(data)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

In [ ]:
df_train = gen_df(dir_dict['train'])
df_val = gen_df(dir_dict['val'])
df_test = gen_df(dir_dict['test'])

In [ ]:
df_train['live'].value_counts()

In [ ]:
df_train['file_path'][0]

In [ ]:
df_train['live'][0]

In [ ]:
df_train_0 = df_train[df_train['live']==0][:1223]
df_train_1 = df_train[df_train['live']==1][:1223]
df_train_balanced = pd.concat([df_train_0, df_train_1]).reset_index(drop=True)
df_train_balanced = df_train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
df_train_balanced

In [ ]:
df_train_balanced['live'].value_counts()

In [ ]:
df_val['live'].value_counts()

In [ ]:
df_val_0 = df_val[df_val['live']==0][:405]
df_val_1 = df_val[df_val['live']==1][:405]
df_val_balanced = pd.concat([df_val_0, df_val_1]).reset_index(drop=True)
df_val_balanced = df_val_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
df_val_balanced['live'].value_counts()

In [ ]:
df_test['live'].value_counts()

In [ ]:
df_test_0 = df_test[df_test['live']==0][:314]
df_test_1 = df_test[df_test['live']==1][:314]
df_test_balanced = pd.concat([df_test_0, df_test_1]).reset_index(drop=True)
df_test_balanced = df_test_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
df_test_balanced['live'].value_counts()

In [ ]:

class FASDataset(Dataset):
    
    def __init__(self, df, transforms=None):
        self.df = df
        self.transforms = transforms
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['file_path']
        label = self.df.iloc[idx]['live']
        label = torch.tensor(label).unsqueeze(0)
        image = Image.open(img_path)
        image = np.array(image)
        
        if self.transforms is not None:
            image = self.transforms(image=image)['image']
        
        return image, label

In [ ]:
train_dataset = FASDataset(df_train, train_transform)
val_dataset = FASDataset(df_val, val_transform)
test_dataset = FASDataset(df_test, test_transform)

dataloader_train = DataLoader(train_dataset, batch_size=32, shuffle=True)
dataloader_val = DataLoader(val_dataset, batch_size=32, shuffle=True)
dataloader_test = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
img, label = next(iter(dataloader_train))
img.size(), label.size()

**Define Model**

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('using \'{}\' device'.format(device))

In [ ]:
class SpoofNet(nn.Module):
    def __init__(self):
        super(SpoofNet, self).__init__()
        # Load pretrained MobileNetV2
        self.pretrained_net = mobilenet_v2(pretrained=True)
        self.features = self.pretrained_net.features
        
        # Adding the extra layers
        self.conv2d = nn.Conv2d(1280, 32, kernel_size=(3, 3), padding=1)  # Adjust input channels if needed
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.features(x)
        x = self.conv2d(x)
        x = self.relu(x)
        x = self.dropout1(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.sigmoid(x)
        return x

# Instantiate the model and print the summary
model = SpoofNet()
model.to(device)

In [ ]:
num_epochs = 40
learning_rate = 5e-5

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCELoss()
scheduler = ReduceLROnPlateau(
    optimizer, factor=0.2, patience=3, verbose=True, 
    threshold=0.005, min_lr=5e-7,
)

# dict to store history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_accuracy': [],
    'val_accuracy': [],
    'learning_rate': [],
}

best_val_loss = 0.0
save_dir = '/kaggle/working/'

# define checkpoint paths
cont_filepath = os.path.join(save_dir, "mobilenetv2-epoch_{}.pt")
best_filepath = os.path.join(save_dir, "mobilenetv2-best.pt")

def save_checkpoint(state, is_best, filename):
    torch.save(state, filename)
    if is_best:
        torch.save(state, best_filepath)


In [ ]:
for epoch in range(num_epochs):
    print('epoch: {}/{}'.format(epoch+1, num_epochs))
    print('-----------------------')
    model.train()
    running_loss = 0.0
    train_total = 0
    train_correct = 0
    prog_bar_train = tqdm(dataloader_train, desc='training')
    for inputs, labels in prog_bar_train:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        predicted = (outputs > 0.5).int()
        loss = criterion(outputs, labels.float())
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        # update progress
        prog_bar_train.set_postfix({'acc': round(train_correct / train_total, 2)})
        
    train_acc = 100 * train_correct / train_total
    avg_train_loss = running_loss / len(dataloader_train)

    history['train_loss'].append(avg_train_loss)
    history['train_accuracy'].append(train_acc)
    
    # validate the model --------------------------------------------------
    model.eval()
    running_val_loss = 0.0
    val_total = 0
    val_correct = 0
    prog_bar_val = tqdm(dataloader_val, desc='validating')
    with torch.no_grad():
        for inputs, labels in prog_bar_val:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels.float())
            running_val_loss += loss.item()
            predicted = (outputs > 0.5).int()
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            # update progress
            prog_bar_val.set_postfix({'acc': round(val_correct / val_total, 4)})
    
     
    avg_val_loss = running_val_loss / len(dataloader_val)
    scheduler.step(avg_val_loss)
    
    val_acc = 100 * val_correct / val_total

    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(val_acc)
    
    
    # Check for improvement ---------------------------
    if epoch == 0:
        best_val_loss = avg_val_loss
        is_best = True
    else:
        is_best = avg_val_loss < best_val_loss
        best_val_loss = min(avg_val_loss, best_val_loss)
    
    checkpoint_filepath = cont_filepath.format(epoch+1)
    print('saving checkpoint: {}'.format(checkpoint_filepath))
    save_checkpoint(
        {'epoch': epoch + 1,
         'state_dict': model.state_dict(),
         'optimizer': optimizer.state_dict(),},
        is_best,
        checkpoint_filepath
    )

    current_lr = optimizer.param_groups[0]['lr']
    history['learning_rate'].append(current_lr)
    
    
    # print loss and accuracy
    print(f'Epoch: {epoch+1}/{num_epochs}'),
    print('Loss/train: {}'.format(avg_train_loss))
    print('Loss/val: {}'.format(avg_val_loss))
    print('Acc/train: {}%'.format(train_acc))
    print('Acc/val: {}%'.format(val_acc))
    print('current lr: {}'.format(current_lr))


In [ ]:
try:
    # Plot the training history
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_accuracy'], label='Train Accuracy')
    plt.plot(history['val_accuracy'], label='Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.show()
except:
    print("exception while plotting results")

In [ ]:
try:   
    plt.figure(figsize=(12, 4))


    plt.subplot(1, 2, 2)
    plt.plot(history['learning_rate'], label='Learning Rate')
#     plt.plot(history['val_accuracy'], label='Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.legend()

    plt.show()
except:
    print("exception while plotting results")

**Evaluation**

In [ ]:
pretrain_weight = '/kaggle/input/spoofnet/pytorch/antispoof/1/mobilenetv2-best.pt'

In [ ]:
check_point = torch.load(pretrain_weight)

model_dict = check_point['state_dict']
epoch_ = check_point['epoch']

# # Load the updated state dictionary into the model
model.load_state_dict(model_dict)

model.to(device)
model.eval()
criterion = nn.BCELoss()


In [ ]:
test_loss = 0.0
correct_predictions = 0
total_predictions = 0
prog_bar_test = tqdm(dataloader_test, desc='Testing')

with torch.no_grad():
    for inputs, labels in prog_bar_test:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels.float())
        test_loss += loss.item()
        
        predicted = (outputs > 0.5).int()
        
        correct_predictions += (predicted == labels).sum().item()
        total_predictions += labels.size(0)
        prog_bar_test.set_postfix({
            'accuracy': correct_predictions / total_predictions * 100,
        })
        
# Calculate average loss and accuracy
test_loss /= len(dataloader_test)
accuracy = correct_predictions / total_predictions * 100

print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {accuracy:.2f}%')

In [ ]:
test_loss = 0.0
correct_predictions = 0
total_predictions = 0
prog_bar_test = tqdm(dataloader_test, desc='Testing')

with torch.no_grad():
    for inputs, labels in prog_bar_test:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels.float())
        test_loss += loss.item()
        
        predicted = (outputs > 0.5).int()
        
        correct_predictions += (predicted == labels).sum().item()
        total_predictions += labels.size(0)
        prog_bar_test.set_postfix({
            'accuracy': correct_predictions / total_predictions * 100,
        })
        
# Calculate average loss and accuracy
test_loss /= len(dataloader_test)
accuracy = correct_predictions / total_predictions * 100

print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {accuracy:.2f}%')